<a href="https://colab.research.google.com/github/Pawan-model/Huggingface-Experiment/blob/main/Chapter_06_Tokenizer_library/Byte_Pair_Encoding_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers

In [ ]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [ ]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained('gpt2')


In [ ]:
from collections import defaultdict
word_freqs=defaultdict(int)
for text in corpus:
  words_with_offset=tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
  new_words=[word for word,offset in words_with_offset]
  for word in new_words:
    word_freqs[word]+=1
print(word_freqs)

In [ ]:
alphabets=[]
for word in word_freqs.keys():
  for letter in word:
    if letter not in alphabets:
      alphabets.append(letter)
alphabets.sort()
print(alphabets)

In [ ]:
vocab = ["<|endoftext|>"] + alphabets.copy()
splits={word:[c for c in word] for word in word_freqs.keys()}

In [ ]:
def compute_pair_freqs(splits):
  pair_freqs=defaultdict(int)
  for word,freq in word_freqs.items():
    split=splits[word]
    if len(split)==1:
      continue
    for i in range(len(split)-1):
      pair=(split[i],split[i+1])
      pair_freqs[pair]+=freq
  return pair_freqs


In [ ]:
pair_freqs = compute_pair_freqs(splits)

for i, key in enumerate(pair_freqs.keys()):
    print(f"{key}: {pair_freqs[key]}")
    if i >= 5:
        break

In [ ]:
best_pair=""
max_freq=None
for pair,freq in pair_freqs.items():
  if max_freq is None or max_freq<freq:
    best_pair=pair
    max_freq=freq
print(best_pair,max_freq)


In [ ]:
def merge_pair(a,b,splits):
  for word,freq in word_freqs.items():
    split=splits[word]
    if len(split)==1:
      continue
    i=0
    while i<len(split)-1:
      if split[i]== a and split[i+1]==b:
        split=split[:i]+[a+b]+split[i+2:]
      else:
        i+=1
    splits[word]=split
  return splits
